## System Preparation (Topology)
We strip the interactive water selection by using -water none and let GROMACS assign protons with -ignh.


In [1]:
import nglview as nv

view = nv.show_structure_file("protein.pdb")
view

NGLWidget()

In [2]:
# Generate topology. 
# -water none prevents adding solvent topology.
# -ignh ignores existing hydrogens and rebuilds them based on the force field.
!gmx pdb2gmx -f protein.pdb -o conf.gro -p topol.top -ff amber99sb-ildn -water none -ignh

                     :-) GROMACS - gmx pdb2gmx, 2026.2 (-:

Executable:   /usr/local/gromacs/bin/gmx
Data prefix:  /usr/local/gromacs
Working dir:  /home/igris/CIU
Command line:
  gmx pdb2gmx -f protein.pdb -o conf.gro -p topol.top -ff amber99sb-ildn -water none -ignh

Using the Amber99sb-ildn force field in directory amber99sb-ildn.ff

going to rename amber99sb-ildn.ff/aminoacids.r2b
Opening force field file /usr/local/gromacs/share/gromacs/top/amber99sb-ildn.ff/aminoacids.r2b

going to rename amber99sb-ildn.ff/dna.r2b
Opening force field file /usr/local/gromacs/share/gromacs/top/amber99sb-ildn.ff/dna.r2b

going to rename amber99sb-ildn.ff/rna.r2b
Opening force field file /usr/local/gromacs/share/gromacs/top/amber99sb-ildn.ff/rna.r2b
Reading protein.pdb...
Read 'IGG2A INTACT ANTIBODY - MAB231; IGG2A INTACT ANTIBODY - MAB231', 10214 atoms

Analyzing pdb file
Splitting chemical chains based on TER records or chain id changing.

There are 4 chains and 0 blocks of water and 1316 residues 

## Box Setup (No Solvation)
We create a box. Even though pbc=no in our MDP, GROMACS requires a box vector. We make it large enough to prevent any accidental boundary issues.

In [3]:
# Create a large bounding box (10.0 nm padding)
# This ensures the box is >20nm in every dimension, 
# allowing a 5.0nm cutoff without periodic image interactions.
!gmx editconf -f conf.gro -o box.gro -c -d 10.0

                     :-) GROMACS - gmx editconf, 2026.2 (-:

Executable:   /usr/local/gromacs/bin/gmx
Data prefix:  /usr/local/gromacs
Working dir:  /home/igris/CIU
Command line:
  gmx editconf -f conf.gro -o box.gro -c -d 10.0

Note that major changes are planned in future for editconf, to improve usability and utility.
Read 20160 atoms
Volume: 503.853 nm^3, corresponds to roughly 226700 electrons
No velocities found
    system size : 10.689 14.755 13.125 (nm)
    center      : -0.045 -1.750  0.909 (nm)
    box vectors :  6.582  7.677 10.064 (nm)
    box angles  :  88.05  92.35  97.23 (degrees)
    box volume  : 503.85               (nm^3)
    shift       : 15.390 19.127 15.654 (nm)
new center      : 15.345 17.378 16.562 (nm)
new box vectors : 30.689 34.755 33.125 (nm)
new box angles  :  90.00  90.00  90.00 (degrees)
new box volume  :35331.00               (nm^3)

If the molecule rotates the actual distance will be smaller. You might want
to use a cubic box instead, or why not try a d

## Write MDP Files
This is the most critical step. We configure vacuum-specific parameters and a linear temperature ramp.

In [4]:
%%writefile em.mdp
integrator  = steep
nsteps      = 5000
emtol       = 100.0
nstlog      = 500
nstxout-compressed = 500

; VACUUM CONSTRAINTS: GROMACS requires pbc=xyz for Verlet lists.
; We use a large box and set cutoffs < half the box size to prevent self-interaction.
pbc         = xyz
coulombtype = cut-off
rcoulomb    = 5.0   ; Must be smaller than half the smallest box dimension
rvdw        = 5.0   ; Must be smaller than half the smallest box dimension

Writing em.mdp


In [5]:
%%writefile nvt_ciu.mdp
integrator  = md
nsteps      = 500000  ; 1 ns (dt=0.002). 
dt          = 0.002
nstlog      = 1000
nstxout-compressed = 5000 ; Save frame every 10 ps

; VACUUM
pbc         = xyz
coulombtype = cut-off
rcoulomb    = 5.0
rvdw        = 5.0

; TEMPERATURE (Simulated Annealing)
; Ramps from 300K to 1000K over 1000 ps (1 ns)
annealing       = single
annealing-npoints = 2
annealing-time  = 0 1000
annealing-temp  = 300 1000

; Thermostat
tc-grps     = System
tau-t       = 0.1
ref-t       = 300  ; Ignored when annealing is active

; Dynamics
constraints = none  ; Allow full flexibility for unfolding

Writing nvt_ciu.mdp


## Run the Simulation
We run minimization, then the heating ramp.

In [6]:
# 1. Energy Minimization
!gmx grompp -f em.mdp -c box.gro -p topol.top -o em.tpr -maxwarn 2
!gmx mdrun -deffnm em

# 2. Gas-Phase Unfolding (NVT)
!gmx grompp -f nvt_ciu.mdp -c em.gro -p topol.top -o nvt_ciu.tpr -maxwarn 2
!gmx mdrun -deffnm nvt_ciu

                      :-) GROMACS - gmx grompp, 2026.2 (-:

Executable:   /usr/local/gromacs/bin/gmx
Data prefix:  /usr/local/gromacs
Working dir:  /home/igris/CIU
Command line:
  gmx grompp -f em.mdp -c box.gro -p topol.top -o em.tpr -maxwarn 2

Setting the LD random seed to -405021218

Generated 2145 of the 2145 non-bonded parameter combinations
Generating 1-4 interactions: fudge = 0.5

Generated 2145 of the 2145 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'Protein_chain_A'

Excluding 3 bonded neighbours molecule type 'Protein_chain_B'

Excluding 3 bonded neighbours molecule type 'Protein_chain_C'

Excluding 3 bonded neighbours molecule type 'Protein_chain_D'

NOTE 1 [file topol.top, line 38]:
  System has non-zero total charge: 2.000000
  Total charge should normally be an integer. See
  https://manual.gromacs.org/current/user-guide/floating-point.html
  for discussion on how close it should be to an integer.



Analysing residue names:
There are:  1316  

**NOTE 1 (Non-zero charge):** This is exactly what we want. A gas-phase protein must have a net charge (in this case, +2) to simulate the protonation state from mass spectrometry.

**NOTE 2 & 3 (Plain Coulomb cut-off):** GROMACS is warning you that you aren't using PME (Particle Mesh Ewald). This is correct; PME requires a neutralizing background plasma that doesn't exist in a vacuum. You must use a plain cut-off for this simulation.

**NOTE 2 (NVE simulation):** GROMACS prints this because you started with zero initial velocities and are using the annealing feature, which overrides standard thermostatting logic. Ignore this; the output clearly confirms it is applying your 300K -> 1000K annealing schedule.

## Interactive Visualization
We use MDAnalysis to parse the GROMACS files and NGLView to render the interactive 3D trajectory directly in the notebook.

In [7]:
import MDAnalysis as mda
import nglview as nv
import ipywidgets as widgets

# Load the initial structure and the unfolding trajectory
u = mda.Universe('em.gro', 'nvt_ciu.xtc')

# Initialize NGLView
view = nv.show_mdanalysis(u)
view.clear_representations()
view.add_representation('cartoon', colorScheme='sstruc')

# Display the widget
display(widgets.VBox([
    widgets.HTML("<b>Gas-Phase Unfolding Trajectory</b><br><i>Use the play button or slider to watch unfolding.</i>"),
    view, 
]))

# Center the view
view.center()